# ID-guard check (23 Nov): T2b's 7B on Palu and Beirut with the ID guard in fixed_flow_c4.py

A copy of t2b_writer_test.ipynb with the 7B only and these two scenes; everything else is T2b's.


In [1]:
import glob, json, os, shutil, signal, socket, subprocess, sys, threading, time, urllib.request, zipfile

SMOKE_TEST = True               # True: Turkey only, for both writers (about 30 min); False: all ten scenes (about 2-2.5 h)
SMOKE_SCENES = ["palu_tsunami_00000180_post_disaster.png", "beriut_explosion_post_16.png"]
WRITERS = [  # short name, Hugging Face repository, commit, name on the server, sees pictures, server overrides
    ("7b", "AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP", "def8f6c81889ffb68904afdc90c3a1389697d73b",
     "disasterm3-7b", True, None),
]
FLOW = "t2b"                    # t2b_flow.py: step 5's flow on T2b's ten scenes, with the photo tool off
MAX_MODEL_LEN = 32768           # tokens, question + answer = LOCAL_LLM_CONTEXT, as in step 5
PORT = 8000
PACKAGES = ["vllm==0.25.1", "crewai==1.15.2"]   # as in step 5; every other package pinned by pip_constraints.txt
RUN_TIMEOUT_HOURS = 1           # per scene: stop the flow if it runs longer (for example, a model in a loop)
WORK = "/kaggle/working/t2b"
os.makedirs(WORK, exist_ok=True)
T0 = time.time()


def minutes():
    return f"{(time.time() - T0) / 60:5.1f} min"

In [2]:
try:
    socket.create_connection(("huggingface.co", 443), timeout=5).close()
    INTERNET = "on"
    print("Internet is on: used only to download the two models and the packages, before the flow runs.")
except OSError:
    raise RuntimeError("Internet is off. T2b downloads its two models and packages: turn it on and run again.")

Internet is on: used only to download the two models and the packages, before the flow runs.


Find the two inputs. If Kaggle kept an uploaded zip as a zip, it is unpacked into `/kaggle/working/unzipped/`.

In [3]:
def find_dir(marker, what, zip_name=None):
    hits = sorted(glob.glob(f"/kaggle/input/**/{marker}", recursive=True))
    if not hits and zip_name:
        for z in glob.glob(f"/kaggle/input/**/{zip_name}", recursive=True):
            with zipfile.ZipFile(z) as f:
                f.extractall(f"/kaggle/working/unzipped/{os.path.basename(z)[:-4]}")
        hits = sorted(glob.glob(f"/kaggle/working/unzipped/**/{marker}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"{what} is missing. Add it with + Add Input.")
    return os.path.dirname(hits[0])


RAG_DIR = find_dir("guideline_index.py", "The offline-rag-kr2 dataset", "offline-rag-kr2*.zip")
DT_INPUT = find_dir("t2b_flow.py", "The disasteller-offline-t2b dataset (make_t2b_upload.py)", "disasteller-offline-t2b*.zip")
for name in ("RAG_DIR", "DT_INPUT"):
    print(f"{name:10} {globals()[name]}")
code = json.load(open(f"{DT_INPUT}/disasteller_manifest.json"))
print("\nDisasTeller code: packed", code["made"], "| git commit", code["git_commit"],
      "| uncommitted changes:", code["uncommitted_changes_in_disasteller_offline"])

RAG_DIR    /kaggle/input/datasets/tamannamou123/offline-rag-kr2
DT_INPUT   /kaggle/input/datasets/tamannamou123/disasteller-offline-t2b

DisasTeller code: packed 2026-11-23 07:11:25 | git commit 1a5f38094 | uncommitted changes: True


Put the DisasTeller files where its code expects them: each file in `disasteller_manifest.json` is found by its
name, checked against its sha256 and copied to `/kaggle/working/disasteller/`, as in step 5. DisasTeller then runs
from that copy.

In [4]:
import hashlib


def sha256(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()


DT_DIR = "/kaggle/working/disasteller"
shutil.rmtree(DT_DIR, ignore_errors=True)
FOUND_AT, missing = {}, []
for rel, digest in sorted(code["sha256"].items()):
    name = os.path.basename(rel)
    candidates = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True)
                        + glob.glob(f"/kaggle/working/unzipped/**/{name}", recursive=True))
    match = next((c for c in candidates if sha256(c) == digest), None)
    if match is None:
        missing.append(f"{rel} (files with this name: {candidates or 'none'})")
        continue
    dest = os.path.join(DT_DIR, rel)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.copy2(match, dest)
    FOUND_AT[rel] = match
if missing:
    raise FileNotFoundError("Missing or changed in the disasteller-offline-t2b dataset: " + "; ".join(missing))
shutil.copy2(f"{DT_INPUT}/disasteller_manifest.json", DT_DIR)
print(f"All {len(FOUND_AT)} files are in {DT_DIR}, each with the sha256 in the manifest.")

# The scenes and their photos, from the checked copy
SCENES = json.load(open(os.path.join(DT_DIR, "t2b_scenes", "scenes.json"), encoding="utf-8"))
if SMOKE_TEST:
    SCENES = {s: SCENES[s] for s in SMOKE_SCENES}
PHOTOS = {(scene, role): os.path.join(DT_DIR, "t2b_scenes", "photos", name)
          for scene, info in SCENES.items() for role, name in (("before", info["before_image"]), ("after", scene))}
assert all(os.path.isfile(p) for p in PHOTOS.values()), "a scene photo is missing"
print(f"{len(SCENES)} scene(s):", ", ".join(f"{s} ({i['disaster']}, rows: {i['kind']})" for s, i in SCENES.items()))

# This session's copy of run_with_netlog.py also knows the flow "t2b" (the dataset's file stays as it is)
NETLOG = os.path.join(DT_DIR, "run_with_netlog.py")
netlog_text = open(NETLOG, encoding="utf-8").read()
OLD_FLOWS = '"fixed": "fixed_flow.py"}'
assert netlog_text.count(OLD_FLOWS) == 1, "run_with_netlog.py changed: check its FLOWS before adding t2b"
with open(NETLOG, "w", encoding="utf-8") as f:
    f.write(netlog_text.replace(OLD_FLOWS, '"fixed": "fixed_flow.py", "t2b": "t2b_flow.py"}'))
print("run_with_netlog.py (this session's copy) knows the flow 't2b': t2b_flow.py")

All 47 files are in /kaggle/working/disasteller, each with the sha256 in the manifest.
2 scene(s): palu_tsunami_00000180_post_disaster.png (tsunami, rows: patch), beriut_explosion_post_16.png (explosion, rows: outline)
run_with_netlog.py (this session's copy) knows the flow 't2b': t2b_flow.py


In [5]:
def gpu_memory():
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip().splitlines()
    return [l.strip() for l in out]


GPUS = gpu_memory()
print(GPUS)
assert len(GPUS) >= 2, "Set Accelerator: GPU T4 x2. The 7B is split across two GPUs."

['Tesla T4, 0 MiB, 15360 MiB', 'Tesla T4, 0 MiB, 15360 MiB']


Install vLLM and CrewAI from the internet, with every package at step 5's version (`pip_constraints.txt`, made
from step 5's pip log), so the two runs have the same packages.

In [6]:
t = time.time()
CONSTRAINTS = os.path.join(DT_DIR, "t2b_scenes", "pip_constraints.txt")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-c", CONSTRAINTS, *PACKAGES], capture_output=True, text=True)
with open(f"{WORK}/pip_install_log.txt", "w") as f:
    f.write(r.stdout + "\n" + r.stderr)
print(r.stdout[-1500:])
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise RuntimeError("pip install failed: see t2b/pip_install_log.txt")
VERSIONS = json.loads(subprocess.run([sys.executable, "-c", (
    "import json, importlib.metadata as m\n"
    "v = {}\n"
    "for p in ('vllm', 'crewai', 'torch', 'transformers', 'openai', 'numpy', 'pillow', 'huggingface_hub'):\n"
    "    try: v[p] = m.version(p)\n"
    "    except Exception: v[p] = None\n"
    "print(json.dumps(v))")], capture_output=True, text=True).stdout)
STEP5 = {"vllm": "0.25.1", "crewai": "1.15.2", "torch": "2.11.0+cu130", "transformers": "5.20.2", "openai": "2.54.0",
         "numpy": "2.1.3", "pillow": "13.0.0"}
print(f"Installed in {(time.time() - t) / 60:.1f} min:", VERSIONS)
print("Different from step 5:", {k: (VERSIONS.get(k), v) for k, v in STEP5.items() if VERSIONS.get(k) != v} or "none")

.3 lance-namespace-0.13.0 lance-namespace-urllib3-client-0.13.0 lancedb-0.30.0 lark-1.2.2 llguidance-1.7.6 llvmlite-0.47.0 lm-format-enforcer-0.11.3 loguru-0.7.3 mcp-1.26.0 mistral_common-1.13.1 model-hosting-container-standards-0.1.16 msgspec-0.22.0 numba-0.65.0 nvidia-cuda-nvcc-13.2.86 nvidia-cuda-tileiras-13.2.86 nvidia-cudnn-frontend-1.28.0 nvidia-cutlass-dsl-4.5.2 nvidia-cutlass-dsl-libs-base-4.5.2 nvidia-cutlass-dsl-libs-cu13-4.5.2 nvidia-nvvm-13.2.86 nvtx-0.2.15 onnxruntime-1.31.1 openai-harmony-0.0.8 opentelemetry-exporter-otlp-1.42.1 opentelemetry-exporter-otlp-proto-common-1.42.1 opentelemetry-exporter-otlp-proto-grpc-1.42.1 opentelemetry-exporter-otlp-proto-http-1.42.1 opentelemetry-proto-1.42.1 opentelemetry-semantic-conventions-ai-0.5.1 outlines_core-0.2.14 partial-json-parser-0.2.1.1.post7 pdfminer.six-20260107 pdfplumber-0.11.10 pendulum-3.3.0 portalocker-2.7.0 posthog-5.4.0 prometheus-fastapi-instrumentator-8.1.0 protobuf-6.33.6 pybase64-1.5.1 pycountry-26.2.16 pydantic

**The writers, one after the other.** For each: download the model at its pinned commit, start vLLM on this machine
only (`127.0.0.1`), split across the two T4s in half precision, as in step 5; two short tests through
`local_model.py`; then the flow on every scene, each in its own process under `run_with_netlog.py`, and the pass rules
(`check_run.py`). The server is then stopped and the model deleted, to make room for the next.

In [7]:
from huggingface_hub import HfApi, snapshot_download

KEY = ("Local model:", "DisasTeller mode", "DisasTeller flow", "Guideline index loaded", "Photo tool",
       "Labelled map saved", "Local model call failed", "Local model gave no answer", "Model message shortened",
       "Section asked again", "Section kept", "Number guard", "Grades found", "Report finished", "Fixed flow finished",
       "Network log", "Outside this computer", "Gemini libraries", "Main script finished", "Traceback", "No C4 facts",
       "Set C4_SCENE", "must hold only", "expert_spec changed")
SMOKE_CODE = ("import json, local_model\n"
              "print(json.dumps({'text': local_model.ask_text('You are a test.', 'In one sentence: what is a building "
              "damage grade?', max_tokens=80), 'fixed flow': local_model.ask_text('You are a test.', 'Reply with the "
              "single word OK.', max_tokens=10)}))")


def model_root(size_gb):
    for root in ("/kaggle/tmp", "/tmp"):
        try:
            os.makedirs(root, exist_ok=True)
            if shutil.disk_usage(root).free / 1e9 > size_gb + 3:
                return os.path.join(root, "t2b_models")
        except OSError:
            pass
    raise RuntimeError(f"No disk has {size_gb + 3:.0f} GB free for the model")


def download(repo, commit):
    info = HfApi().model_info(repo, revision=commit, files_metadata=True)
    size = sum(s.size or 0 for s in info.siblings) / 1e9
    out = os.path.join(model_root(size), repo.split("/")[1])
    t = time.time()
    snapshot_download(repo, revision=commit, local_dir=out)
    shutil.rmtree(os.path.join(out, ".cache"), ignore_errors=True)   # only the download's bookkeeping
    bad = [s.rfilename for s in info.siblings if not os.path.isfile(os.path.join(out, s.rfilename))
           or (s.size is not None and os.path.getsize(os.path.join(out, s.rfilename)) != s.size)]
    assert not bad, f"{repo}: download incomplete: {bad}"
    generation = json.load(open(os.path.join(out, "generation_config.json"))) if os.path.isfile(
        os.path.join(out, "generation_config.json")) else {}
    print(f"[{minutes()}] {repo} at {info.sha[:12]}: {len(info.siblings)} files, {size:.1f} GB in "
          f"{(time.time() - t) / 60:.1f} min -> {out}; generation_config: {generation}")
    return out, info.sha, generation


def start_server(model_dir, served, vision, overrides, log_path):
    vllm_cli = shutil.which("vllm")
    cmd = ([vllm_cli, "serve", model_dir] if vllm_cli
           else [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", model_dir])
    cmd += ["--served-model-name", served, "--host", "127.0.0.1", "--port", str(PORT),
            "--tensor-parallel-size", "2", "--dtype", "half", "--enforce-eager",
            "--max-model-len", str(MAX_MODEL_LEN), "--gpu-memory-utilization", "0.92", "--max-num-seqs", "8"]
    if vision:      # as in step 5; a text model takes no picture settings
        cmd += ["--limit-mm-per-prompt", json.dumps({"image": 1}),
                "--mm-processor-kwargs", json.dumps({"max_pixels": 1344 * 1344})]
    if overrides:   # the same decoding for both writers (first cell)
        cmd += ["--override-generation-config", json.dumps(overrides)]
    env = dict(os.environ, HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", VLLM_NO_USAGE_STATS="1", DO_NOT_TRACK="1")
    print(" ".join(cmd))
    log = open(log_path, "w")
    server = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True)
    for _ in range(180):  # up to 30 minutes
        if server.poll() is not None:
            break
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5)
            text = open(log_path, errors="replace").read()
            lines = [l[-200:] for l in text.splitlines()
                     if any(k in l for k in ("Model loading took", "KV cache", "concurrency", "sampling param",
                                             "generation config", "generation_config"))]
            print(f"[{minutes()}] vLLM is ready")
            for l in lines:
                print("  ", l)
            return server, lines
        except OSError:
            time.sleep(10)
    print(open(log_path, errors="replace").read()[-5000:])
    raise RuntimeError(f"vLLM did not start: its log is above and in {log_path}")


def stop_server(server):
    """The whole process group (vLLM's two GPU workers too), then wait until both GPUs are free again."""
    for sig, wait in ((signal.SIGTERM, 60), (signal.SIGKILL, 30)):
        try:
            os.killpg(os.getpgid(server.pid), sig)
            server.wait(timeout=wait)
            break
        except (ProcessLookupError, subprocess.TimeoutExpired):
            pass
    for _ in range(24):
        used = [int(g.split(",")[1].split()[0]) for g in gpu_memory()]
        if max(used) < 1024:
            break
        time.sleep(5)
    print(f"[{minutes()}] server stopped; GPU memory now {gpu_memory()}")


def writer_env(served, call_log):
    env = {k: v for k, v in os.environ.items() if k != "DISASTELLER_ONLINE"}
    env.update(LOCAL_LLM_PROVIDER="hosted_vllm", LOCAL_LLM_MODEL=served, LOCAL_LLM_BASE_URL=f"http://127.0.0.1:{PORT}/v1",
               LOCAL_LLM_CONTEXT=str(MAX_MODEL_LEN), GUIDELINE_BACKEND="numpy", GUIDELINE_INDEX_DIR=RAG_DIR,
               HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", CUDA_VISIBLE_DEVICES="", PYTHONDONTWRITEBYTECODE="1",
               PYTHONIOENCODING="utf-8", PYTHONUNBUFFERED="1", CREWAI_DISABLE_TELEMETRY="true",
               CREWAI_TRACING_ENABLED="false", CREWAI_DISABLE_VERSION_CHECK="true", LOCAL_LLM_CALL_LOG=call_log)
    return env


def smoke(name, served):
    r = subprocess.run([sys.executable, "-c", SMOKE_CODE], cwd=DT_DIR, capture_output=True, text=True,
                       env=writer_env(served, f"{WORK}/{name}/smoke_llm_calls.jsonl"))
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-3000:])
        raise RuntimeError(f"The smoke test of {name} failed")
    answers = json.loads(r.stdout.strip().splitlines()[-1])
    for label, answer in answers.items():
        print(f"  {label}: {answer[:300]}")
        if answer.startswith("An error occurred"):
            raise RuntimeError(f"The {label} test of {name} failed: see t2b/{name}/vllm_server.log")
    return answers


def run_scenes(name, served):
    local = os.path.join(DT_DIR, "Disaster_image", "Local_image")
    results = {}
    for scene, info in SCENES.items():
        # The flow's checks want only this pair in Local_image/ (the after-photo also gets the labelled map)
        shutil.rmtree(local, ignore_errors=True)
        os.makedirs(local)
        shutil.copy2(PHOTOS[(scene, "before")], os.path.join(local, f"1_before_{info['before_image']}"))
        shutil.copy2(PHOTOS[(scene, "after")], os.path.join(local, f"2_after_{scene}"))
        run_dir = f"{WORK}/{name}/run_{os.path.splitext(scene)[0]}"
        shutil.rmtree(run_dir, ignore_errors=True)
        os.makedirs(run_dir)
        env = writer_env(served, "llm_calls.jsonl")   # in run_dir
        env.update(DISASTELLER_FLOW=FLOW, C4_SCENE=scene)
        print(f"\n[{minutes()}] ===== {name}: {scene} ({info['disaster']}, rows: {info['kind']})")
        start = time.time()
        proc = subprocess.Popen([sys.executable, os.path.join(DT_DIR, "run_with_netlog.py")], cwd=run_dir, env=env,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                                errors="replace")
        timer = threading.Timer(RUN_TIMEOUT_HOURS * 3600, proc.kill)
        timer.start()
        for line in proc.stdout:
            if any(k in line for k in KEY):
                print(f"[{minutes()}] {line.rstrip()[:220]}")
        proc.wait()
        timer.cancel()
        check = subprocess.run([sys.executable, os.path.join(DT_DIR, "check_run.py"), run_dir, RAG_DIR,
                                "--internet", INTERNET], capture_output=True, text=True)
        results[scene] = {"run_dir": run_dir, "flow_exit_code": proc.returncode, "check_exit_code": check.returncode,
                          "minutes": round((time.time() - start) / 60, 2)}
        print(f"[{minutes()}] {scene}: flow exit code {proc.returncode}, pass rules exit code {check.returncode}, "
              f"{results[scene]['minutes']} min")
        print("\n".join(l for l in check.stdout.splitlines() if l.startswith(("PASS", "FAIL", "Verdict", "PASSED"))))
    return results

In [8]:
RESULTS, WRITER_INFO = {}, {}
for name, repo, commit, served, vision, overrides in WRITERS:
    print(f"\n[{minutes()}] ########## writer {name}: {repo}")
    os.makedirs(f"{WORK}/{name}", exist_ok=True)
    model_dir, sha, generation = download(repo, commit)
    assert sha == commit, f"{repo}: got commit {sha}, not {commit}"
    t = time.time()
    server, server_lines = start_server(model_dir, served, vision, overrides, f"{WORK}/{name}/vllm_server.log")
    load = (time.time() - t) / 60
    smoke_answers = smoke(name, served)
    t = time.time()
    RESULTS[name] = run_scenes(name, served)
    WRITER_INFO[name] = {"repo": repo, "commit": sha, "served_name": served, "sees_pictures": vision,
                         "generation_config": generation, "server_overrides": overrides,
                         "load_minutes": round(load, 1), "scenes_minutes": round((time.time() - t) / 60, 1),
                         "server_log_lines": server_lines, "smoke_tests": smoke_answers}
    stop_server(server)
    shutil.rmtree(model_dir, ignore_errors=True)   # room for the next model
    passed = sum(r["flow_exit_code"] == 0 and r["check_exit_code"] == 0 for r in RESULTS[name].values())
    print(f"[{minutes()}] writer {name}: {passed} of {len(RESULTS[name])} scenes passed the five pass rules, "
          f"{WRITER_INFO[name]['scenes_minutes']} min for the scenes")


[  2.8 min] ########## writer 7b: AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

[  4.1 min] AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP at def8f6c81889: 8 files, 16.6 GB in 1.3 min -> /kaggle/tmp/t2b_models/disasterm3-qwen2.5vl7b-l0s2-mergedFP; generation_config: {'bos_token_id': 151643, 'do_sample': True, 'eos_token_id': [151645, 151643], 'pad_token_id': 151643, 'repetition_penalty': 1.05, 'temperature': 1e-06, 'transformers_version': '5.16.1'}
/usr/local/bin/vllm serve /kaggle/tmp/t2b_models/disasterm3-qwen2.5vl7b-l0s2-mergedFP --served-model-name disasterm3-7b --host 127.0.0.1 --port 8000 --tensor-parallel-size 2 --dtype half --enforce-eager --max-model-len 32768 --gpu-memory-utilization 0.92 --max-num-seqs 8 --limit-mm-per-prompt {"image": 1} --mm-processor-kwargs {"max_pixels": 1806336}
[  8.0 min] vLLM is ready
   (Worker_TP1 pid=283) INFO 11-23 01:19:37 [model_runner.py:302] Model loading took 7.91 GiB and 66.442899 seconds
   (Worker_TP0 pid=282) INFO 11-23 01:19:37 [model_runner.py:302] Model loading took 7.91 GiB and 66.451566 seconds
   (Worker_TP0 p

In [9]:
info = {"smoke_test": SMOKE_TEST, "internet": INTERNET, "flow": FLOW, "scenes": list(SCENES), "results": RESULTS,
        "writers": WRITER_INFO, "minutes": round((time.time() - T0) / 60, 1), "max_model_len": MAX_MODEL_LEN,
        "versions": VERSIONS, "code_packed": code["made"], "code_git_commit": code["git_commit"],
        "code_uncommitted_changes": code["uncommitted_changes_in_disasteller_offline"], "code_files_found_at": FOUND_AT}
with open(f"{WORK}/t2b_notebook_info.json", "w") as f:
    json.dump(info, f, indent=2)
zip_path = shutil.make_archive(f"/kaggle/working/t2b_{'smoke' if SMOKE_TEST else 'full'}_internet_{INTERNET}", "zip", WORK)
print(json.dumps({k: v for k, v in info.items() if k not in ("code_files_found_at", "writers")}, indent=1)[:3000])
print("Zip:", zip_path)

{
 "smoke_test": true,
 "internet": "on",
 "flow": "t2b",
 "scenes": [
  "palu_tsunami_00000180_post_disaster.png",
  "beriut_explosion_post_16.png"
 ],
 "results": {
  "7b": {
   "palu_tsunami_00000180_post_disaster.png": {
    "run_dir": "/kaggle/working/t2b/7b/run_palu_tsunami_00000180_post_disaster",
    "flow_exit_code": 0,
    "check_exit_code": 0,
    "minutes": 6.35
   },
   "beriut_explosion_post_16.png": {
    "run_dir": "/kaggle/working/t2b/7b/run_beriut_explosion_post_16",
    "flow_exit_code": 0,
    "check_exit_code": 0,
    "minutes": 7.04
   }
  }
 },
 "minutes": 21.6,
 "max_model_len": 32768,
 "versions": {
  "vllm": "0.25.1",
  "crewai": "1.15.2",
  "torch": "2.11.0+cu130",
  "transformers": "5.20.2",
  "openai": "2.54.0",
  "numpy": "2.1.3",
  "pillow": "13.0.0",
  "huggingface_hub": "1.33.0"
 },
 "code_packed": "2026-11-23 07:11:25",
 "code_git_commit": "1a5f38094",
 "code_uncommitted_changes": true
}
Zip: /kaggle/working/t2b_smoke_internet_on.zip


**After the run:** the outputs go into `Offline_RAG_KR2/results/t2b/` (the smoke test into `results/t2b_smoke/`),
and the executed notebook over this one. Then `t2b_score.ipynb` scores both writers by the rule above, on the PC.